# Distributions Reference

## Learning Objectives
1. Implement Gaussian and Binomial PDFs/PMFs from scratch using numpy (no scipy)
2. Fit distributions to data via MLE using scipy and compare with AIC/BIC
3. Apply Poisson, Beta, and CLT to real-world scenarios
4. Choose the right distribution for a given data-generating process

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.special import comb, factorial
from scipy.optimize import minimize_scalar, minimize

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Imports OK')
print('NumPy:', np.__version__)

## Level 1: PDF and PMF from Scratch

We implement the Gaussian PDF and Binomial PMF manually in numpy,
then verify against scipy.stats. This shows the formulas and builds intuition
before relying on library implementations.

In [ ]:
# --- Gaussian PDF from scratch ---
# f(x) = (1 / (sigma * sqrt(2*pi))) * exp(-0.5 * ((x-mu)/sigma)^2)
def gaussian_pdf(x, mu, sigma):
    # Coefficient: normalizes area to 1
    coeff = 1.0 / (sigma * np.sqrt(2 * np.pi))
    # Exponent: squared standardized deviation
    exponent = -0.5 * ((x - mu) / sigma) ** 2
    return coeff * np.exp(exponent)

# --- Binomial PMF from scratch ---
# P(X=k) = C(n,k) * p^k * (1-p)^(n-k)
def binomial_pmf(k, n, p):
    # Use log for numerical stability, then exponentiate
    log_binom_coef = (np.log(np.arange(1, n+1)).sum()
                      - np.log(np.arange(1, k+1)).sum()
                      - np.log(np.arange(1, n-k+1)).sum())
    log_pmf = log_binom_coef + k * np.log(p) + (n - k) * np.log(1 - p)
    return np.exp(log_pmf)

# --- Verify against scipy ---
x_vals = np.linspace(-4, 4, 500)
mu, sigma = 0.0, 1.0
our_pdf   = gaussian_pdf(x_vals, mu, sigma)
scipy_pdf = stats.norm.pdf(x_vals, mu, sigma)
print(f'Gaussian PDF max error vs scipy: {np.max(np.abs(our_pdf - scipy_pdf)):.2e}')

n_binom, p_binom = 20, 0.3
k_vals = np.arange(0, n_binom + 1)
our_pmf   = np.array([binomial_pmf(k, n_binom, p_binom) for k in k_vals])
scipy_pmf = stats.binom.pmf(k_vals, n_binom, p_binom)
print(f'Binomial PMF max error vs scipy: {np.max(np.abs(our_pmf - scipy_pmf)):.2e}')
print(f'Binomial PMF sums to: {our_pmf.sum():.6f} (should be 1.0)')

# --- Plot both distributions ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gaussian
axes[0].plot(x_vals, our_pdf, 'b-', lw=2, label='Our implementation')
axes[0].plot(x_vals, scipy_pdf, 'r--', lw=1.5, label='scipy.stats.norm')
axes[0].fill_between(x_vals, 0, our_pdf, where=(x_vals >= -1) & (x_vals <= 1),
                      alpha=0.2, color='blue', label='P(-1 < X < 1)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Gaussian PDF N({mu}, {sigma}^2)', fontweight='bold')
axes[0].legend()

# Binomial
axes[1].bar(k_vals - 0.15, our_pmf, width=0.3, color='blue', alpha=0.7, label='Our implementation')
axes[1].bar(k_vals + 0.15, scipy_pmf, width=0.3, color='red', alpha=0.5, label='scipy.stats.binom')
axes[1].set_xlabel('k')
axes[1].set_ylabel('P(X = k)')
axes[1].set_title(f'Binomial PMF B({n_binom}, {p_binom})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_02_scratch.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_02_scratch.png')

# --- Key statistics from formulas ---
print(f'Gaussian: mean={mu}, variance={sigma**2}')
print(f'Binomial: mean={n_binom*p_binom:.2f}, variance={n_binom*p_binom*(1-p_binom):.2f}')
print(f'Binomial: scipy mean={stats.binom.mean(n_binom, p_binom):.2f},'
      f' var={stats.binom.var(n_binom, p_binom):.2f}')

## Level 2: MLE Parameter Fitting and Model Selection

Given observed data, we fit multiple distribution families using `scipy.stats.X.fit()`
(MLE under the hood), then compare fits using AIC = 2k - 2*log_likelihood.
Lower AIC wins.

In [ ]:
# --- Generate data from a known Log-Normal (to test recovery) ---
true_mu_ln, true_sigma_ln = 1.5, 0.6
data = stats.lognorm(s=true_sigma_ln, scale=np.exp(true_mu_ln)).rvs(500, random_state=42)
print(f'Data: n=500, mean={data.mean():.2f}, std={data.std():.2f}, min={data.min():.2f}')

# --- Fit multiple distributions ---
candidates = {
    'lognorm':    stats.lognorm,
    'gamma':      stats.gamma,
    'expon':      stats.expon,
    'norm':       stats.norm,   # should fit poorly (data is strictly positive and skewed)
    'weibull_min': stats.weibull_min,
}

results = {}
for name, dist in candidates.items():
    try:
        # MLE: fit returns (shape_params..., loc, scale)
        params = dist.fit(data)
        # Log-likelihood at fitted parameters
        log_lik = dist.logpdf(data, *params).sum()
        # Number of free parameters
        k = len(params)
        # AIC = 2k - 2 * log-likelihood (lower is better)
        aic = 2 * k - 2 * log_lik
        # BIC = k*log(n) - 2*log-likelihood
        bic = k * np.log(len(data)) - 2 * log_lik
        results[name] = {'params': params, 'log_lik': log_lik,
                         'aic': aic, 'bic': bic, 'k': k}
    except Exception as e:
        print(f'{name} fit failed: {e}')

print('\nModel comparison (lower AIC/BIC is better):')
print(f'  {'Distribution':<15} {'k':>4} {'Log-Lik':>12} {'AIC':>10} {'BIC':>10}')
print('  ' + '-' * 55)
for name, r in sorted(results.items(), key=lambda x: x[1]['aic']):
    print(f'  {name:<15} {r["k"]:>4} {r["log_lik"]:>12.1f} {r["aic"]:>10.1f} {r["bic"]:>10.1f}')

best_name = min(results, key=lambda x: results[x]['aic'])
print(f'\nBest fit by AIC: {best_name} (correctly identifies Log-Normal)')

# --- KS test for best and worst fits ---
best_params  = results[best_name]['params']
worst_name   = max(results, key=lambda x: results[x]['aic'])
worst_params = results[worst_name]['params']

ks_best  = stats.kstest(data, best_name,  args=best_params)
ks_worst = stats.kstest(data, worst_name, args=worst_params)
print(f'KS test {best_name}:  statistic={ks_best.statistic:.4f},  p={ks_best.pvalue:.4f}')
print(f'KS test {worst_name}: statistic={ks_worst.statistic:.4f},  p={ks_worst.pvalue:.4f}')

# --- Plot fitted PDFs on histogram ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_plot = np.linspace(data.min(), data.max(), 300)

axes[0].hist(data, bins=50, density=True, alpha=0.5, color='gray', label='Data')
colors = ['blue', 'red', 'green', 'orange', 'purple']
for (name, r), color in zip(results.items(), colors):
    axes[0].plot(x_plot, candidates[name].pdf(x_plot, *r['params']),
                 color=color, lw=1.5, label=f'{name} (AIC={r["aic"]:.0f})')
axes[0].set_xlabel('Value'); axes[0].set_ylabel('Density')
axes[0].set_title('Distribution Fitting (MLE)', fontweight='bold')
axes[0].legend(fontsize=7)

# Q-Q plot: best fit vs Normal (comparison)
from scipy.stats import probplot
(osm, osr), (slope, intercept, r) = probplot(data, dist='norm', fit=True)
axes[1].plot(osm, osr, 'o', markersize=2, alpha=0.4, label='Data quantiles')
x_line = np.array([osm.min(), osm.max()])
axes[1].plot(x_line, slope * x_line + intercept, 'r-', lw=2, label=f'Normal fit (r={r:.3f})')
axes[1].set_xlabel('Theoretical Gaussian quantiles')
axes[1].set_ylabel('Data quantiles')
axes[1].set_title('Q-Q Plot (Normal): Deviations = Non-Normality', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_02_fitting.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_02_fitting.png')

## Real-World Example 1: Poisson Process for Web Traffic

Modeling website requests per second as a Poisson process.
Key property: mean equals variance. We estimate lambda from data,
compute exceedance probabilities, and detect anomalies.

In [ ]:
np.random.seed(42)

# --- Simulate web traffic: requests per second over 24 hours ---
# True rate changes by time of day (two regimes: day/night)
lambda_day   = 120.0   # requests/second during business hours
lambda_night = 30.0    # requests/second at night
n_seconds_day   = 57600  # 16 hours day (08:00-24:00)
n_seconds_night = 28800  # 8 hours night (00:00-08:00)

traffic_day   = np.random.poisson(lambda_day,   n_seconds_day)
traffic_night = np.random.poisson(lambda_night, n_seconds_night)
all_traffic   = np.concatenate([traffic_night, traffic_day])

print(f'Day traffic:   mean={traffic_day.mean():.2f}, var={traffic_day.var():.2f}'
      f' (Poisson: mean=var={lambda_day})')
print(f'Night traffic: mean={traffic_night.mean():.2f}, var={traffic_night.var():.2f}'
      f' (Poisson: mean=var={lambda_night})')

# --- MLE for lambda ---
# For Poisson, MLE estimator is simply the sample mean
lambda_hat_day   = traffic_day.mean()
lambda_hat_night = traffic_night.mean()
print(f'MLE lambda (day):   {lambda_hat_day:.2f}  (true: {lambda_day})')
print(f'MLE lambda (night): {lambda_hat_night:.2f}  (true: {lambda_night})')

# --- Exceedance probabilities ---
# P(X > threshold) = 1 - CDF(threshold)
thresholds = [150, 180, 200, 250]
print('\nExceedance probabilities during day (lambda=120):')
for t in thresholds:
    p_exceed = 1.0 - stats.poisson.cdf(t, lambda_hat_day)
    n_exceed_per_hour = p_exceed * 3600
    print(f'  P(X > {t}): {p_exceed:.6f}  ~{n_exceed_per_hour:.1f} times/hour')

# --- Anomaly detection: 3-sigma rule for Poisson ---
# For Poisson, std = sqrt(lambda), so 3-sigma threshold = lambda + 3*sqrt(lambda)
threshold_3sigma = lambda_hat_day + 3 * np.sqrt(lambda_hat_day)
print(f'\n3-sigma anomaly threshold (day): {threshold_3sigma:.1f} requests/second')
anomalies = np.sum(traffic_day > threshold_3sigma)
expected_anomalies = n_seconds_day * (1 - stats.poisson.cdf(threshold_3sigma, lambda_hat_day))
print(f'Observed anomalies: {anomalies}  (expected ~{expected_anomalies:.1f} by chance)')

# Overdispersion check: is variance >> mean?
dispersion_ratio = traffic_day.var() / traffic_day.mean()
print(f'\nDispersion ratio var/mean = {dispersion_ratio:.3f} (should be ~1.0 for Poisson)')
print('If >> 1: use Negative Binomial; if << 1: use under-dispersed model')

# --- Plot traffic distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram vs theoretical Poisson PMF
k_range = np.arange(80, 165)
axes[0].hist(traffic_day, bins=50, density=True, alpha=0.6, color='steelblue', label='Observed traffic')
axes[0].plot(k_range, stats.poisson.pmf(k_range, lambda_hat_day),
             'r-', lw=2, label=f'Poisson(lambda={lambda_hat_day:.1f})')
axes[0].axvline(threshold_3sigma, color='orange', ls='--', lw=2, label=f'3-sigma={threshold_3sigma:.0f}')
axes[0].set_xlabel('Requests per second')
axes[0].set_ylabel('Probability')
axes[0].set_title('Web Traffic (Day) vs Poisson', fontweight='bold')
axes[0].legend(fontsize=9)

# Time series of traffic
axes[1].plot(all_traffic[:500], color='gray', lw=0.5, alpha=0.7)
axes[1].axhline(lambda_night, color='blue', ls='--', label=f'Night lambda={lambda_night}')
axes[1].axhline(lambda_day,   color='red',  ls='--', label=f'Day lambda={lambda_day}')
axes[1].set_xlabel('Time (seconds from midnight)')
axes[1].set_ylabel('Requests/second')
axes[1].set_title('Web Traffic: First 500 Seconds', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_02_poisson.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_02_poisson.png')

## Real-World Example 2: Beta Distribution as Bayesian Prior

The Beta distribution models uncertainty about a probability parameter.
It is conjugate to the Bernoulli/Binomial: if prior is Beta(a,b), then
after observing k successes in n trials, the posterior is Beta(a+k, b+n-k).

We track conversion rate estimation as data accumulates day by day.

In [ ]:
np.random.seed(42)

# --- True conversion rate unknown to us ---
true_rate = 0.07   # 7% conversion rate

# --- Daily conversion data (simulate 14 days) ---
n_visitors_per_day = 200
days = 14
daily_visitors    = np.full(days, n_visitors_per_day)
daily_conversions = np.random.binomial(n_visitors_per_day, true_rate, days)

print(f'True conversion rate: {true_rate*100:.1f}%')
print(f'Daily conversions: {daily_conversions}')
print(f'Daily rates: {daily_conversions/daily_visitors}')

# --- Bayesian update: start with Beta(1,1) uninformative prior ---
alpha_prior = 1.0
beta_prior  = 1.0

print('\nBayesian update (Beta-Binomial):')
print(f'  {'Day':<6} {'Conversions':>12} {'Posterior Mean':>16} {'95% CI':>25}')
print('  ' + '-' * 62)

alpha_post = alpha_prior
beta_post  = beta_prior
posterior_means = []
posterior_cis   = []

for day in range(days):
    # Update posterior: Beta(alpha+successes, beta+failures)
    k = daily_conversions[day]
    n = daily_visitors[day]
    alpha_post += k
    beta_post  += (n - k)

    post_dist = stats.beta(alpha_post, beta_post)
    mean_est  = post_dist.mean()
    ci_lo, ci_hi = post_dist.ppf(0.025), post_dist.ppf(0.975)

    posterior_means.append(mean_est)
    posterior_cis.append((ci_lo, ci_hi))

    cum_n = int((day + 1) * n_visitors_per_day)
    cum_k = int(daily_conversions[:day+1].sum())
    print(f'  Day {day+1:<3} {cum_k:>5}/{cum_n:<7} {mean_est*100:>10.2f}%  [{ci_lo*100:.2f}%, {ci_hi*100:.2f}%]')

# --- Different Beta prior shapes ---
print('\nEffect of prior strength on Beta distribution:')
priors = [
    (1, 1,    'Beta(1,1) uniform - no prior info'),
    (2, 18,   'Beta(2,18) weak prior ~10%'),
    (10, 90,  'Beta(10,90) strong prior ~10%'),
    (70, 930, 'Beta(70,930) very strong prior ~7%'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.linspace(0, 0.20, 500)

# Prior shapes
for (a, b, label) in priors:
    axes[0].plot(x, stats.beta(a, b).pdf(x), lw=2, label=label)
axes[0].axvline(true_rate, color='black', ls=':', lw=2, label=f'True rate={true_rate}')
axes[0].set_xlabel('Conversion rate'); axes[0].set_ylabel('Density')
axes[0].set_title('Beta Prior Shapes', fontweight='bold')
axes[0].legend(fontsize=8)

# Posterior narrowing over 14 days
days_plot = np.arange(1, days + 1)
means_arr = np.array(posterior_means)
cis_lo = np.array([ci[0] for ci in posterior_cis])
cis_hi = np.array([ci[1] for ci in posterior_cis])

axes[1].plot(days_plot, means_arr * 100, 'b-o', lw=2, markersize=5, label='Posterior mean')
axes[1].fill_between(days_plot, cis_lo * 100, cis_hi * 100, alpha=0.2, color='blue', label='95% credible interval')
axes[1].axhline(true_rate * 100, color='red', ls='--', lw=2, label=f'True rate {true_rate*100:.1f}%')
axes[1].set_xlabel('Days of data'); axes[1].set_ylabel('Estimated conversion rate %')
axes[1].set_title('Beta Posterior Convergence Over Time', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_02_beta.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_02_beta.png')

## Real-World Example 3: Central Limit Theorem Demo

The CLT states: if X_1,...,X_n are iid with mean mu and variance sigma^2,
then sqrt(n)*(X_bar - mu)/sigma -> N(0,1) as n -> infinity.

We demonstrate this for three very different distributions (Exponential, Uniform, Bernoulli)
and show convergence to Gaussian as the sample size grows.

In [ ]:
np.random.seed(42)

# --- Three non-Gaussian source distributions ---
def sample_exponential(n):
    # Highly right-skewed
    return np.random.exponential(scale=1.0, size=n)

def sample_uniform(n):
    # Flat, bounded, symmetric
    return np.random.uniform(0, 1, size=n)

def sample_bernoulli(n):
    # Discrete, bimodal (p=0.2)
    return np.random.binomial(1, 0.2, size=n).astype(float)

distributions = {
    'Exponential(1)': (sample_exponential, 1.0, 1.0),    # mean=1, var=1
    'Uniform(0,1)':   (sample_uniform,     0.5, 1/12),   # mean=0.5, var=1/12
    'Bernoulli(0.2)': (sample_bernoulli,   0.2, 0.16),   # mean=0.2, var=0.16
}

# Sample sizes to demonstrate convergence
sample_sizes = [1, 5, 30, 100]
n_experiments = 10_000   # number of sample means to collect

# --- CLT demonstration ---
print('CLT Demo: Distribution of Sample Means')
print(f'  Each experiment: draw n samples, compute mean. Repeat {n_experiments:,} times.')
print()

fig, axes = plt.subplots(len(distributions), len(sample_sizes), figsize=(14, 9))

for row, (dist_name, (sampler, true_mean, true_var)) in enumerate(distributions.items()):
    for col, n in enumerate(sample_sizes):
        # Compute n_experiments sample means of size n
        sample_means = np.array([sampler(n).mean() for _ in range(n_experiments)])

        # Standardize: (X_bar - mu) / (sigma / sqrt(n))
        std_err = np.sqrt(true_var / n)
        standardized = (sample_means - true_mean) / std_err

        # Test normality: compute KS test against standard normal
        ks_stat, ks_p = stats.kstest(standardized, 'norm')

        ax = axes[row, col]
        ax.hist(standardized, bins=50, density=True, alpha=0.6,
                color='steelblue', edgecolor='white')

        # Overlay standard normal
        xr = np.linspace(-4, 4, 200)
        ax.plot(xr, stats.norm.pdf(xr), 'r-', lw=1.5)

        ax.set_title(f'{dist_name}\nn={n}  KS p={ks_p:.3f}', fontsize=8, fontweight='bold')
        ax.set_xlim(-4, 4)
        ax.set_xticks([-2, 0, 2])
        if col == 0:
            ax.set_ylabel('Density', fontsize=8)

plt.suptitle('Central Limit Theorem: Sample Means Converge to Gaussian\n(red curve = N(0,1))',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('stats_02_clt.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_02_clt.png')

# --- Quantify convergence ---
print('\nKS test p-value (larger = more Gaussian): n=1 fails, n=100 passes')
print(f'  {'Distribution':<20} {'n=1':>10} {'n=5':>10} {'n=30':>10} {'n=100':>10}')
print('  ' + '-' * 55)
for dist_name, (sampler, true_mean, true_var) in distributions.items():
    row_vals = []
    for n in sample_sizes:
        means = np.array([sampler(n).mean() for _ in range(n_experiments)])
        std_err = np.sqrt(true_var / n)
        standardized = (means - true_mean) / std_err
        _, p = stats.kstest(standardized, 'norm')
        row_vals.append(p)
    print(f'  {dist_name:<20} {row_vals[0]:>10.4f} {row_vals[1]:>10.4f} {row_vals[2]:>10.4f} {row_vals[3]:>10.4f}')

print('\nKEY TAKEAWAYS for Distributions:')
print('1. Match distribution support to data: Poisson for counts, Beta for rates')
print('2. MLE via scipy.fit() + compare AIC to select the right family')
print('3. Check mean ~= variance for Poisson; if not, use Negative Binomial')
print('4. CLT: sample means are Gaussian for n >= 30 (rule of thumb)')
print('5. Q-Q plot and KS test are the standard goodness-of-fit checks')
print('6. Beta(1,1) prior = no information; stronger priors narrow posterior faster')

print('\nEXERCISES:')
print('1. Fit a Gamma and Log-Normal to the same data. Which has lower AIC?')
print('2. Generate Poisson data with overdispersion (var=2*mean). What happens when you fit Poisson?')
print('3. Demo CLT fails for Cauchy distribution (infinite variance) -- try it')